# Sum of Square Spectral Amplification Resource Estimation

This notebook demonstrates how to use QDK Chemistry to construct a Sum of Squares Spectral Amplification (SOSSA) block-encoding circuit and perform resource estimation using QRE v3.
The goal is to enable users to understand and further improve on the fault-tolerant resource requirements of state-of-the-art quantum algorithms.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md), you will need to install the `jupyter` and `qre` extras to run this notebook:

```bash
pip install 'qdk-chemistry[jupyter,qre]'
```

In [1]:
# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

## Load a DFTHC factorized Hamiltonian for H2

Load DFTHC tensors from a `.dfthc.json` file and construct a `FactorizedHamiltonianContainer`.
The JSON uses the same field names as the C++ serialization format (`one_body_integrals`, `u_matrices`, `w_matrices`, `wb_matrix`).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from qdk_chemistry.algorithms.controlled_circuit_mapper.sossa_mapper import (
    InnerPrepareMapper,
    OuterPrepareMapper,
    SelectMapper,
    SOSSAMapper,
)
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.block_encoding.sossa import SOSSABuilder
from qdk_chemistry.data import BasisSet, FactorizedHamiltonianContainer, Hamiltonian, Orbitals, OrbitalType, Shell
from qdk_chemistry.data.controlled_unitary import ControlledUnitary
from qdk_chemistry.data.unitary_representation.containers.sossa import SOSSAContainer

HAM_JSON = Path("h2_factorized_r1_b2_c1.hamiltonian.json")

ham = Hamiltonian.from_json(HAM_JSON.read_text())
fh = ham.get_container()
N = fh.get_num_orbitals()
R, B, C = fh.get_num_ranks(), fh.get_num_bases(), fh.get_num_copies()
print(f"Loaded from {HAM_JSON}: N={N}, R={R}, B={B}, C={C}")

Factorized Hamiltonian: N=2, R=1, B=1, C=1


## Generate SOSSA Circuit

In [3]:
# Step 1: SOSSABuilder → UnitaryRepresentation
builder = SOSSABuilder(quantum_walk=True)
unitary_rep = builder.run(fh)
container = unitary_rep.get_container()

# Step 2: SOSSAMapper → Circuit
controlled_unitary = ControlledUnitary(unitary=unitary_rep, control_indices=[0])
mapper = SOSSAMapper(
    outer_prepare_mapper=OuterPrepareMapper(algorithm="dense_pure"),
    inner_prepare_mapper=InnerPrepareMapper(algorithm="direct"),
    select_mapper=SelectMapper(multiplexed_rotation="direct"),
)
circuit = mapper.run(controlled_unitary)
lambda_sos = container.normalization
print(f"SOSSA normalization Λ = {lambda_sos:.6f}")
print(f"Circuit built successfully")

SOSSA normalization Λ = 1.122596
Circuit built successfully


## Verify QPE simulation

In [ ]:
from qdk_chemistry.algorithms.phase_estimation.iterative_phase_estimation import IterativePhaseEstimation
from qdk_chemistry.data import AlgorithmRef
from qdk_chemistry.data.circuit import Circuit, QsharpFactoryData
from qdk_chemistry.utils.qsharp import QSHARP_UTILS

# Hartree-Fock state |1100⟩ as trial state (2 electrons in lowest orbitals)
num_system_qubits = 2 * N
hf_bitstring = [1, 1] + [0] * (num_system_qubits - 2)
state_prep_params = {
    "rowMap": list(range(num_system_qubits - 1, -1, -1)),
    "stateVector": [0.0] * (2**num_system_qubits),
    "expansionOps": [],
    "numQubits": num_system_qubits,
}
# Set HF amplitude: |1100⟩ in big-endian = index 12
hf_index = sum(b << (num_system_qubits - 1 - i) for i, b in enumerate(hf_bitstring))
state_prep_params["stateVector"][hf_index] = 1.0

qsharp_factory = QsharpFactoryData(
    program=QSHARP_UTILS.StatePreparation.MakeStatePreparationCircuit,
    parameter=state_prep_params,
)
qsharp_op = QSHARP_UTILS.StatePreparation.MakeStatePreparationOp(state_prep_params)
state_prep = Circuit(qsharp_factory=qsharp_factory, qsharp_op=qsharp_op)

from qdk_chemistry.algorithms.phase_estimation.standard_phase_estimation import StandardPhaseEstimation

# Run standard (QFT-based) QPE with SOSSA walk operator
num_bits_std = 5
std_qpe = StandardPhaseEstimation(shots=3)
std_qpe.settings().set("qpe_circuit_builder", AlgorithmRef(
    "qpe_circuit_builder", "qdk_standard",
    num_bits=num_bits_std,
    controlled_circuit_mapper=AlgorithmRef(
        "controlled_circuit_mapper", "sossa",
        outer_prepare_algorithm="dense_pure",
        inner_prepare_algorithm="direct",
        select_algorithm="direct",
    ),
    unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa", quantum_walk=True),
))
std_qpe.settings().set(
    "circuit_executor",
    AlgorithmRef("circuit_executor", "qdk_sparse_state_simulator"),
)

import matplotlib.pyplot as plt

# Collect QPE phases over multiple runs
num_runs = 30
phases = []
for _ in range(num_runs):
    r = std_qpe.run(state_preparation=state_prep, factorized_hamiltonian=fh)
    phases.append(r.phase_fraction)

# Count occurrences of each measured phase
from collections import Counter
counts = Counter(phases)
labels = [f"{p:.4f}" for p in sorted(counts)]
values = [counts[p] for p in sorted(counts)]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(labels, values, color="steelblue", edgecolor="white")
ax.set_xlabel("Measured phase φ")
ax.set_ylabel("Counts")
ax.set_title(f"Standard QPE phase distribution ({num_runs} runs, {num_bits_std} bits)")
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.3, str(v), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

# Print energy from most common phase
best_phase = counts.most_common(1)[0][0]
E_best = lambda_sos * np.cos(2 * np.pi * best_phase)
print(f"Most frequent phase: {best_phase:.4f}  →  E = Λ·cos(2πφ) = {E_best:.6f}")
print(f"Energy gap estimate: {lambda_sos - E_best:.6f}")

QPE phase: 0.000000
QPE raw energy (Λ·cos 2πφ): 1.122596
Energy gap estimate: 0.000000


## QRE v3 Resource Estimation (Standard QPE)

Build the full standard QPE circuit and use `circuit.estimate()` (QRE v3) for logical resource counts.

In [6]:
from qdk_chemistry.algorithms.phase_estimation.circuit_builder.standard_builder import QdkStandardQpeCircuitBuilder

# Build the full standard QPE circuit for resource estimation
num_bits_re_std = 10

std_builder = QdkStandardQpeCircuitBuilder(
    num_bits=num_bits_re_std,
    controlled_circuit_mapper=AlgorithmRef(
        "controlled_circuit_mapper", "sossa",
        outer_prepare_algorithm="dense_pure",
        inner_prepare_algorithm="direct",
        select_algorithm="direct",
    ),
    unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa", quantum_walk=True),
)

std_circuits = std_builder.run(
    state_preparation=state_prep,
    factorized_hamiltonian=fh,
)
std_qpe_circuit = std_circuits[0]

# QRE v3 resource estimation via qsharp.estimate()
std_qpe_estimate = std_qpe_circuit.estimate()
df_std_qpe = pd.DataFrame(
    std_qpe_estimate.logical_counts.items(),
    columns=["Logical Estimate", "Counts"],
)
print(f"Standard QPE logical resource counts ({num_bits_re_std}-bit precision):")
display(df_std_qpe)

Standard QPE logical resource counts (10-bit precision):


,Logical Estimate,Counts
0,numQubits,27
1,tCount,8211
2,rotationCount,30798
3,rotationDepth,30746
4,cczCount,136059
5,ccixCount,0
6,measurementCount,15355


In [ ]:
# QRE v3 via qdk.qre.estimate() (advanced path)
try:
    from qdk.qre import estimate, plot_estimates
    from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

    app = std_qpe_circuit.get_qre_application()
    architecture = Majorana(error_rate=1e-5)
    isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())

    qre_result = estimate(app, architecture, isa_query, max_error=0.01, name="SOSSA QPE")
    print("QRE v3 (qdk.qre) estimation complete:")
    plot_estimates(qre_result, figsize=(6, 4), runtime_unit="ms")
except (ImportError, RuntimeError, TypeError) as e:
    print(f"qdk.qre not available (expected in dev): {e}")
    print("Falling back to qsharp.estimate() results above.")

TypeError: estimate() missing 2 required positional arguments: 'architecture' and 'isa_query'